<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/03-rag/02-hybrid-and-reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Retrieval & Reranking

**Goal:** Combine keyword and vector retrieval, add a reranker, measure whether each earns its cost.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `requests`, `sentence-transformers`, `numpy`, `rank-bm25` are used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" requests sentence-transformers numpy rank-bm25

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 1.6 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


## The problem: one retriever can't cover a real query stream

Notebook 01 ended on a split verdict: vector search nails paraphrases but fumbles exact identifiers, and a dumb substring grep does the opposite. That's not an academic curiosity. Your production query stream contains *both* kinds at once. Someone pastes `Sec-WebSocket-Accept` and expects the exact section; someone else asks "how do I keep a connection open both ways?" and expects the same WebSocket RFC. No single retriever serves both users well.

This notebook fixes that by stacking components, but every stage you add costs latency and money, so the real skill is knowing which ones earn their place. We build up in three moves, cheapest first:

| Stage | What it is | What it catches | Cost | Verdict |
|---|---|---|---|---|
| **BM25** | keyword scoring (TF-IDF, grown up) | exact identifiers, rare tokens, error strings | ~free, no API | baseline that refuses to die |
| **Vector search** | bi-encoder cosine (from nb 01) | paraphrases, conceptual questions | ~free locally | complements BM25, doesn't replace it |
| **Hybrid (RRF)** | fuse the two ranked lists by rank | robustness across *both* query types | ~free | nearly all upside, add it |
| **Reranker** | read query+passage together, score relevance | mention-vs-answer distinction the first two miss | expensive (N calls/query) | only if an eval says it earns it |

The through-line: **add stages cheapest-first, and make each one prove it earns its cost before it stays.** The final section measures exactly that.

## Setup: corpus, chunks, embeddings

Same corpus and section-aware chunks as notebook 01 (the cells are repeated because every notebook here is self-contained), plus the vector index rebuilt the same way. The chunker is a compact copy; notebook 03 is where chunking itself gets unpacked.


In [3]:
# Download the shared corpus: ten IETF RFCs, cached to data/rfc/.
# Every notebook in this repo that needs the corpus includes this cell --
# self-containment over DRY, so each notebook runs top-to-bottom on its own.
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc number -> raw text
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # cached: skip the download on re-runs
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

RFC  791  IP                 94,892 chars
RFC  793  TCP               172,710 chars
RFC 1035  DNS               122,549 chars
RFC 2616  HTTP/1.1          422,279 chars
RFC 4271  BGP               222,702 chars
RFC 5321  SMTP              225,929 chars
RFC 6455  WebSocket         162,067 chars
RFC 6749  OAuth 2.0         163,498 chars
RFC 7540  HTTP/2            209,580 chars
RFC 9110  HTTP Semantics    502,907 chars


In [4]:
# Compact copy of the cleaning + section-aware chunking. Notebook 03 (chunking)
# is where this is built up and its tradeoffs explained; here it's just a given.
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # page footers
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # page headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # long section: paragraph-pack it, carrying the heading along
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

1588 chunks across 10 RFCs


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)

def vector_search(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = embeddings @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

## BM25: the keyword baseline that refuses to die

BM25 is TF-IDF with better math: it scores a chunk by how many query terms it contains, discounted by how common each term is corpus-wide and by chunk length. No neural network, no training, no GPU, and it's still the retrieval baseline that embedding papers have to beat.

Notebook 01 ended with vector search losing to a substring grep on exact identifiers. BM25 is that grep, grown up.


In [6]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'[a-z0-9][a-z0-9._-]*', text.lower())

bm25 = BM25Okapi([tokenize(t) for t in chunk_texts])

def bm25_search(query, k=5):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

def show(results, label):
    print(f'--- {label}')
    for score, i in results:
        first_line = chunk_texts[i].strip().split('\n')[0][:70]
        print(f'  {score:7.3f}  RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})  {first_line}')
    print()

## Where each one wins

Two queries, both retrievers. First an exact identifier, `RST`, the TCP reset flag. Then a paraphrase with zero keyword overlap with the text that answers it.


In [7]:
q1 = 'when does TCP send a RST segment?'
show(bm25_search(q1), f'BM25:   {q1}')
show(vector_search(q1), f'vector: {q1}')

q2 = 'how can a browser keep a two-way conversation open with a server?'
show(bm25_search(q2), f'BM25:   {q2}')
show(vector_search(q2), f'vector: {q2}')

--- BM25:   when does TCP send a RST segment?
   23.437  RFC  793 (TCP)  3.9.  Event Processing
   22.850  RFC  793 (TCP)  3.4.  Establishing a connection
   22.432  RFC 7540 (HTTP/2)  8.3.  The CONNECT Method
   21.271  RFC  793 (TCP)  1822
   18.817  RFC  793 (TCP)  3.4.  Establishing a connection

--- vector: when does TCP send a RST segment?
    0.678  RFC  793 (TCP)  1822
    0.671  RFC  793 (TCP)  3.9.  Event Processing
    0.592  RFC 7540 (HTTP/2)  5.1.  Stream States
    0.590  RFC  793 (TCP)  3.4.  Establishing a connection
    0.586  RFC  793 (TCP)  3.9.  Event Processing

--- BM25:   how can a browser keep a two-way conversation open with a server?
   21.663  RFC 6455 (WebSocket)  1.6.  Security Model
   21.286  RFC 6455 (WebSocket)  4.1.  Client Requirements
   20.645  RFC 2616 (HTTP/1.1)  8.1.2.1 Negotiation
   18.789  RFC 9110 (HTTP Semantics)  16.7.  Upgrade Token Registry
   18.345  RFC 6455 (WebSocket)  4.1.  Client Requirements

--- vector: how can a browser keep a tw

Run this and note the split decision:

- On the **RST query**, BM25 nails it: `rst` is a rare token, so chunks containing it dominate. Vector search retrieves generally-TCP-flavored text, and the exact RST-handling sections may or may not crack the top 5, because the three-letter identifier barely dents a 384-dim embedding.
- On the **two-way conversation query**, BM25 has almost nothing to grab (the WebSocket RFC says "bidirectional" and "full-duplex," not "two-way conversation"), while vector search lands on RFC 6455 easily, because the *meaning* matches even though the words don't.

Neither wins both. Your production query stream contains both kinds: people paste error strings and identifiers, and people ask vague conceptual questions. So you want both retrievers, which raises the question of how to merge two ranked lists.

## Hybrid: reciprocal rank fusion

You can't average the scores: BM25 scores are unbounded and corpus-dependent, cosine scores live in [-1, 1]. They're not on comparable scales, and any weighted sum of them is a lie with a tuning knob. **Reciprocal rank fusion** sidesteps the problem by throwing the scores away and using only the *ranks*: each list contributes `1 / (k + rank)` per document, and you sum. Rank 1 in either list is worth a lot; rank 40 is worth almost nothing; `k=60` (the standard constant) damps the difference between rank 1 and rank 3 so one list can't dominate.


In [8]:
def rrf(rankings, k=60, top_n=20):
    """rankings: list of ranked lists of chunk indices. Returns [(idx, score), ...]."""
    scores = {}
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]

def hybrid_search(query, top_n=20, pool=50):
    v = [i for _, i in vector_search(query, pool)]
    b = [i for _, i in bm25_search(query, pool)]
    return rrf([v, b], top_n=top_n)

for q in [q1, q2]:
    fused = hybrid_search(q, top_n=5)
    show([(s, i) for i, s in fused], f'hybrid: {q}')

--- hybrid: when does TCP send a RST segment?
    0.033  RFC  793 (TCP)  3.9.  Event Processing
    0.032  RFC  793 (TCP)  1822
    0.031  RFC  793 (TCP)  3.4.  Establishing a connection
    0.031  RFC  793 (TCP)  3.4.  Establishing a connection
    0.030  RFC 7540 (HTTP/2)  8.3.  The CONNECT Method

--- hybrid: how can a browser keep a two-way conversation open with a server?
    0.029  RFC 7540 (HTTP/2)  9.1.  Connection Management
    0.029  RFC 6455 (WebSocket)  1.1.  Background
    0.027  RFC 6455 (WebSocket)  4.1.  Client Requirements
    0.027  RFC 2616 (HTTP/1.1)  8.1.2.1 Negotiation
    0.027  RFC 6455 (WebSocket)  1.6.  Security Model



Run this and note that hybrid holds up on *both* queries: the RST sections survive because BM25 ranked them first, the WebSocket sections survive because vector search did. That robustness across query types, not a higher ceiling on any single query, is what hybrid buys.

## Reranking: a second, more expensive opinion

Both retrievers scored each chunk against the query *independently and cheaply*. BM25 never read the sentence, and the bi-encoder compressed query and chunk into vectors before comparing. A **reranker** reads the query and the passage *together* and scores actual relevance. That's much more accurate and much more expensive, which is why it runs on 20 fused candidates, not the whole corpus.

We'll use the model itself as a pointwise reranker: one cheap call per passage, "rate relevance 0–10, digits only." Be clear about what this is: **N API calls per query.** Production systems use a cross-encoder reranker (Cohere Rerank, a hosted API, or a local model) that does the same read-them-together trick at a fraction of the cost and latency. The concept is identical, so learning it this way transfers directly.


In [9]:
def llm_score(query, passage):
    prompt = (
        'Rate how relevant this passage is for answering the query, '
        'on a scale of 0 (irrelevant) to 10 (directly answers it).\n\n'
        f'Query: {query}\n\nPassage:\n{passage[:1500]}\n\n'
        'Reply with a single integer 0-10 and nothing else.'
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=8,
        messages=[{'role': 'user', 'content': prompt}],
    )
    text = resp.choices[0].message.content
    m = re.search(r'\d+', text)
    return min(int(m.group()), 10) if m else 0

def rerank(query, candidate_ids):
    scored = [(llm_score(query, chunk_texts[i]), i) for i in candidate_ids]
    return sorted(scored, key=lambda x: -x[0])

In [10]:
query = 'when does TCP send a RST segment?'

fused = hybrid_search(query, top_n=20)
candidates = [i for i, _ in fused]
reranked = rerank(query, candidates)   # 20 API calls -- watch the wall clock

print('hybrid top-5 (by RRF)          vs   reranked top-5 (by LLM 0-10)')
for (i_h, s_h), (s_r, i_r) in zip(fused[:5], reranked[:5]):
    left = chunk_texts[i_h].strip().split('\n')[0][:32]
    right = chunk_texts[i_r].strip().split('\n')[0][:32]
    print(f'  {s_h:.4f} {left:<34} {s_r:>2}/10 {right}')

hybrid top-5 (by RRF)          vs   reranked top-5 (by LLM 0-10)
  0.0325 3.9.  Event Processing              0/10 3.9.  Event Processing
  0.0320 1822                                0/10 1822
  0.0313 3.4.  Establishing a connection     0/10 3.4.  Establishing a connection
  0.0310 3.4.  Establishing a connection     0/10 3.4.  Establishing a connection
  0.0302 8.3.  The CONNECT Method            0/10 8.3.  The CONNECT Method


Run this and compare the two columns. Typical result on this corpus: the reranker promotes the section that actually *defines* RST behavior over sections that merely *mention* RST a lot. That's exactly the mention-vs-answer distinction that term counting and vector proximity can't see. Sometimes the lists barely differ; that's data too (see below).

## The honest accounting: each stage must earn its cost

Retrieval pipelines accrete stages because each one sounds reasonable. The discipline is to measure what each stage adds in result quality against what it costs in latency and money, on *your* queries.


In [11]:
import time

query = 'how does a server tell a client to retry later?'

t0 = time.perf_counter()
v = vector_search(query, 50)
t1 = time.perf_counter()
b = bm25_search(query, 50)
t2 = time.perf_counter()
fused = rrf([[i for _, i in v], [i for _, i in b]], top_n=20)
t3 = time.perf_counter()
reranked = rerank(query, [i for i, _ in fused])
t4 = time.perf_counter()

print(f'vector search : {(t1 - t0) * 1000:8.1f} ms')
print(f'bm25 search   : {(t2 - t1) * 1000:8.1f} ms')
print(f'rrf fusion    : {(t3 - t2) * 1000:8.1f} ms')
print(f'LLM rerank    : {(t4 - t3) * 1000:8.1f} ms   (20 API calls)')
print()
show([(s, i) for i, s in fused[:5]], 'hybrid top-5')
show(reranked[:5], 'reranked top-5')

vector search :    217.2 ms
bm25 search   :     66.5 ms
rrf fusion    :      0.5 ms
LLM rerank    :   5357.9 ms   (20 API calls)

--- hybrid top-5
    0.032  RFC 2616 (HTTP/1.1)  8.2.4 Client Behavior if Server Prematurely Closes Connection
    0.030  RFC 2616 (HTTP/1.1)  8.1.2.2 Pipelining
    0.030  RFC 7540 (HTTP/2)  8.1.4.  Request Reliability Mechanisms in HTTP/2
    0.028  RFC 9110 (HTTP Semantics)  9.2.2.  Idempotent Methods
    0.028  RFC 5321 (SMTP)  4.5.4.1.  Sending Strategy

--- reranked top-5
    0.000  RFC 2616 (HTTP/1.1)  8.2.4 Client Behavior if Server Prematurely Closes Connection
    0.000  RFC 2616 (HTTP/1.1)  8.1.2.2 Pipelining
    0.000  RFC 7540 (HTTP/2)  8.1.4.  Request Reliability Mechanisms in HTTP/2
    0.000  RFC 9110 (HTTP Semantics)  9.2.2.  Idempotent Methods
    0.000  RFC 5321 (SMTP)  4.5.4.1.  Sending Strategy



Run this and study the shape of the numbers, not the exact values:

- **BM25 and fusion are effectively free:** microseconds to low milliseconds, no API cost. Adding BM25 to a vector pipeline is nearly all upside. It costs nothing measurable and it rescues the whole class of exact-identifier queries. It earns its cost trivially.
- **The LLM rerank is a different animal:** tens of seconds serially (20 round trips) and roughly 10K input tokens per query, free on Groq but a couple of cents at typical paid-API pricing. A production cross-encoder collapses that to tens of milliseconds and fractions of a cent, but even then it's the most expensive stage per candidate. It earns its cost only if final answers improve, which depends on how often your fused top-5 is wrong in ways the reranker fixes.

And that last clause is the trap: you can't settle it by eyeballing two queries. Whether reranking earns its place is an *evaluation* question. You need a set of questions with known answers and a hit-rate to compare with and without. Notebook 04 builds a small one by hand; section 04 makes it rigorous. Resist adding stages until you can measure them.


## Practices & anti-patterns

| ✅ Do | ❌ Anti-pattern |
|---|---|
| Add BM25 alongside vectors; it's ~free and rescues exact-identifier queries | Ship vector-only and lose every acronym/error-code lookup |
| Fuse ranked lists with RRF (rank-based) | Average BM25 and cosine scores (different scales, a lie with a knob) |
| Add a reranker only when an eval shows it earns its latency/cost | Stack rerankers because each stage "sounds reasonable" |
| Rerank a small candidate pool (top-20), not the corpus | Run the expensive cross-encoder over everything |
| Settle "does this stage help?" with a hit-rate, not two queries | Eyeball two examples and declare victory |

(The full stack-wide list lives in [docs/best-practices-and-anti-patterns.md](https://github.com/calmrocks/ai-engineer-notebooks/blob/main/docs/best-practices-and-anti-patterns.md).)

## Exercises

1. **Weighted RRF.** Add a per-list weight to `rrf()` (multiply each list's contribution). Try weighting vectors 2:1 over BM25 and the reverse, on both `q1` and `q2`. Find a query where the weighting flips the top result, then decide whether you could justify that knob to a teammate without an eval.
2. **Batch the reranker.** Rewrite `rerank()` to score all 20 passages in a *single* API call: number the passages in one prompt and ask for a JSON object mapping passage number to score. Measure the latency and cost drop, then check how often its scores disagree with the pointwise version. You're rediscovering the listwise-vs-pointwise tradeoff.
3. **Rerank depth sweep.** Rerank the top 5, 10, and 20 fused candidates for three queries of your choice. At what depth does the final top-3 stop changing? That depth is your real candidate budget; everything past it is spend without effect.
4. **Cheap-model reranker.** Swap the reranker to `openai/gpt-oss-20b` and compare its 0–10 scores against the 120B's on the same 20 candidates. If they produce the same *ranking*, the cheaper model earns the job, the first of many times you'll downshift a pipeline stage after measuring.